In [3]:
#ô code 1 
import pandas as pd


df = pd.read_csv('data/Dataset-Unicauca-Version2-87Atts.csv')
df = df.sample(100000, random_state=42).reset_index(drop=True) #lấy 100k dòng chạy thử và đánh lại số tt chạy từ đầu 
# Lệnh này sẽ gom nhóm và đếm tất cả các loại app có trong file
print(df['ProtocolName'].unique())
print(f"Tổng số ứng dụng khác nhau là: {df['ProtocolName'].nunique()}")



<StringArray>
[          'HTTP',         'GOOGLE',     'HTTP_PROXY',   'HTTP_CONNECT',
      'MICROSOFT',         'AMAZON',  'CONTENT_FLASH',        'YOUTUBE',
          'GMAIL',            'SSL', 'WINDOWS_UPDATE',       'FACEBOOK',
            'MSN',          'SKYPE',        'DROPBOX',     'OFFICE_365',
        'TWITTER',     'CLOUDFLARE',     'TEAMVIEWER',          'YAHOO',
        'NETFLIX',        'IP_ICMP',            'DNS',        'SPOTIFY',
   'MS_ONE_DRIVE',       'WHATSAPP',          'APPLE',      'UBUNTUONE',
   'APPLE_ITUNES',    'GOOGLE_MAPS',           'EBAY',    'SSL_NO_CERT',
      'INSTAGRAM',           'MQTT',         'TIMMEU',       'FTP_DATA',
           'WAZE',           'H323',      'WIKIPEDIA',   'APPLE_ICLOUD',
       'EASYTAXI',         'RADIUS',        'EDONKEY',  'HTTP_DOWNLOAD',
         'DEEZER',            'TOR',            'NTP',    'FTP_CONTROL',
      'WHOIS_DAS',          'OSCAR',       'TELEGRAM',            'SSH',
         'ORACLE',            'BGP']


In [4]:
#ô code 2
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
import numpy as np

# 1. Mã hóa nhãn: Biến 'YOUTUBE', 'FACEBOOK'... thành số 0, 1, 2...
le = LabelEncoder()
df['ProtocolName_Encoded'] = le.fit_transform(df['ProtocolName'])

# 2. Chọn các cột đặc trưng (Features) - Loại bỏ các cột không phải số hoặc không cần thiết
# Mình tạm loại bỏ các cột IP và ID vì chúng dễ làm AI bị "học vẹt"
features = df.select_dtypes(include=[np.number]).columns.tolist()
features = [f for f in features if f not in ['ProtocolName_Encoded', 'L7Protocol']]

X = df[features]
y = df['ProtocolName_Encoded']

# 3. Xử lý giá trị vô hạn (Inf) hoặc lỗi dữ liệu thường gặp trong lưu lượng mạng
X = X.replace([np.inf, -np.inf], np.nan).fillna(0)

# 4. Chia dữ liệu: 80% để học, 20% để kiểm tra
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 5. Chuẩn hóa: Đưa các con số về cùng một thang đo (0-1 hoặc tương đương)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Đã chuẩn bị xong dữ liệu cho {len(features)} đặc trưng và {df['ProtocolName'].nunique()} ứng dụng.")


Đã chuẩn bị xong dữ liệu cho 80 đặc trưng và 54 ứng dụng.


In [5]:
#ô code 3 
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, accuracy_score
from sklearn.preprocessing import LabelEncoder

# 1. Tái mã hóa nhãn riêng cho tập Train và Test để đảm bảo tính liên tục (0, 1, 2...)
# Bước này cực kỳ quan trọng để sửa lỗi "Invalid classes inferred"
le_final = LabelEncoder()
y_train_fixed = le_final.fit_transform(y_train)

# Đối với tập Test, chúng ta chỉ lấy những nhãn mà tập Train đã học được
# Những nhãn nào tập Train không có sẽ bị loại bỏ ở tập Test để không gây lỗi
test_mask = y_test.isin(le_final.classes_)
X_test_fixed = X_test_scaled[test_mask]
y_test_fixed = le_final.transform(y_test[test_mask])

# 2. Khởi tạo mô hình XGBoost
model_baseline = XGBClassifier(n_estimators=100, random_state=42, n_jobs=-1)

# 3. Huấn luyện mô hình
print(f"Đang huấn luyện mô hình cơ sở trên {len(le_final.classes_)} ứng dụng...")
model_baseline.fit(X_train_scaled, y_train_fixed)

# 4. Dự đoán và đánh giá
y_pred = model_baseline.predict(X_test_fixed)
print(f"\n--- KẾT QUẢ HOÀN THÀNH WEEK 3 ---")
print(f"Độ chính xác (Accuracy): {accuracy_score(y_test_fixed, y_pred):.4f}")
print("\nBáo cáo chi tiết (Classification Report):")
print(classification_report(y_test_fixed, y_pred))

Đang huấn luyện mô hình cơ sở trên 54 ứng dụng...

--- KẾT QUẢ HOÀN THÀNH WEEK 3 ---
Độ chính xác (Accuracy): 0.7318

Báo cáo chi tiết (Classification Report):
              precision    recall  f1-score   support

           0       0.73      0.46      0.57       496
           1       0.78      0.39      0.52        54
           2       0.00      0.00      0.00         5
           3       0.00      0.00      0.00        11
           5       0.71      0.37      0.48        82
           6       0.82      0.68      0.74        40
           7       0.00      0.00      0.00         1
           8       0.67      0.50      0.57         4
           9       0.96      0.69      0.81       157
          10       1.00      0.17      0.29         6
          11       0.00      0.00      0.00         7
          13       0.84      0.75      0.79       164
          15       0.00      0.00      0.00         2
          16       0.44      0.07      0.12       222
          17       0.71      

c:\Users\ADMIN\OneDrive\Máy tính\TTCS\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\ADMIN\OneDrive\Máy tính\TTCS\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\ADMIN\OneDrive\Máy tính\TTCS\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capital

In [6]:
#
#ô code 4 
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv1D, MaxPooling1D, Flatten, Dense, Dropout

In [7]:
# ô code 5
import numpy as np
# Chuyển từ 2D sang 3D bằng cách dùng đúng tên biến đã chuẩn hóa ở trên
X_train_cnn = np.expand_dims(X_train_scaled, axis=2)
X_test_cnn = np.expand_dims(X_test_fixed, axis=2) 

print(f"Cấu trúc dữ liệu 3D cho CNN: {X_train_cnn.shape}")

Cấu trúc dữ liệu 3D cho CNN: (80000, 80, 1)


In [8]:
# ô code 6
num_features = X_train_scaled.shape[1] # Tự động lấy số lượng cột (đặc trưng)
num_classes = len(le_final.classes_)   # Tự động lấy số lượng ứng dụng

model = Sequential([
    # Lớp lọc đặc trưng (Convolutional Layer)
    Conv1D(64, 3, activation='relu', input_shape=(num_features, 1)),
    MaxPooling1D(2),
    # Lớp làm phẳng dữ liệu để đưa vào mạng nơ-ron
    Flatten(),
    Dense(128, activation='relu'),
    Dropout(0.3),
    # Lớp đầu ra (Số lượng app thực tế)
    Dense(num_classes, activation='softmax') # Khớp với số lượng app thực tế
])
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
model.summary()

c:\Users\ADMIN\OneDrive\Máy tính\TTCS\.venv\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv1d (Conv1D)                 │ (None, 78, 64)         │           256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d (MaxPooling1D)    │ (None, 39, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 2496)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │       319,616 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 54)             │         6,966 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 326,838 (1.25 MB)

 Trainable params: 326,838 (1.25 MB)

 Non-trainable params: 0 (0.00 B)

In [9]:
# ô code 7
# Sử dụng y_train_fixed và y_test_fixed để khớp với mã hóa của XGBoost ở trên
history = model.fit(
    X_train_cnn, 
    y_train_fixed, 
    epochs=50, 
    batch_size=64,
    validation_data=(X_test_cnn, y_test_fixed)
)

Epoch 1/50
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 6s 4ms/step - accuracy: 0.4625 - loss: 1.6921 - val_accuracy: 0.5185 - val_loss: 1.5347
Epoch 2/50
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 5s 4ms/step - accuracy: 0.5126 - loss: 1.5268 - val_accuracy: 0.5267 - val_loss: 1.4584
Epoch 3/50
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 6s 5ms/step - accuracy: 0.5318 - loss: 1.4660 - val_accuracy: 0.5448 - val_loss: 1.4093
Epoch 4/50
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 6s 5ms/step - accuracy: 0.5462 - loss: 1.4256 - val_accuracy: 0.5609 - val_loss: 1.3875
Epoch 5/50
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 12s 10ms/step - accuracy: 0.5546 - loss: 1.3970 - val_accuracy: 0.5712 - val_loss: 1.3558
Epoch 6/50
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 6s 4ms/step - accuracy: 0.5610 - loss: 1.3759 - val_accuracy: 0.5700 - val_loss: 1.3347
Epoch 7/50
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 5s 4ms/step - accuracy: 0.5676 - loss: 1.3569 - val_accuracy: 0.5784 - val_loss: 1.3178
Epoch 8/50
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 5s 4ms/step - accuracy: 0.5701 - loss: 1.3402 

KeyboardInterrupt: 